# Credit risk | Q3 DPD30 and the proposed limit cut

Finance reported about 6.5% defaults in July and about 12% in September, and proposed a 30% cut to maximum limits. This notebook reconstructs mature DPD30, tests a comparable disbursal window, and assesses the evidence for the proposed policy.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
DATA = ROOT / "data" / "cleaned"
EXTRACT = pd.Timestamp("2026-11-05")
SOURCE_TO_CHANNEL = {
    "meta": "Meta", "facebook ads": "Meta", "facebook": "Meta", "fb": "Meta",
    "instagram": "Meta", "ig": "Meta",
    "google": "Google Ads", "google ads": "Google Ads", "adwords": "Google Ads", "gads": "Google Ads",
    "tiktok": "TikTok", "tt": "TikTok", "referral": "Referral", "ref": "Referral",
    "invite": "Referral", "organic": "Organic", "(direct)": "Direct",
}


users = pd.read_csv(DATA / "users.csv")
loans = pd.read_csv(DATA / "loans.csv", parse_dates=["disbursed_date", "due_date", "last_repayment_date"])
assert loans.user_id.isin(users.user_id).all()

q3 = loans.loc[
    loans.disbursed_date.ge("2026-07-01") & loans.disbursed_date.lt("2026-10-01")
].merge(users[["user_id", "utm_source_normalized"]], on="user_id", validate="many_to_one")
source = q3.utm_source_normalized
q3["channel"] = source.map(SOURCE_TO_CHANNEL).fillna("Unmapped")
q3.loc[source.isna() | source.eq("").fillna(False), "channel"] = "Unattributed"
assert q3.due_date.sub(q3.disbursed_date).dt.days.eq(14).all()

## DPD30 by disbursal month

A loan is mature once its due date plus 30 days is on or before November 5. A mature loan defaults at DPD30 if its principal plus the recorded fee was still unpaid in whole or in part at the cutoff. Use the final repayment date for loans fully repaid by extract, counting full repayment on the cutoff day as on time. The separate unpaid-at-extract rate includes immature loans and illustrates how a different denominator changes the comparison. Date-only fields are used as supplied.

In [2]:
q3["month"] = q3.disbursed_date.dt.to_period("M")
q3["dpd30_date"] = q3.due_date + pd.Timedelta(days=30)
q3["mature"] = q3.dpd30_date.le(EXTRACT)
q3["unpaid_at_extract"] = q3.amount_repaid_egp.lt(q3.principal_egp + q3.fee_egp - .01)
assert q3.loc[~q3.unpaid_at_extract, "last_repayment_date"].notna().all()
full_by_cutoff = ~q3.unpaid_at_extract & q3.last_repayment_date.le(q3.dpd30_date)
q3["dpd30_default"] = q3.mature & ~full_by_cutoff


def credit_summary(frame, group):
    result = frame.groupby(group, observed=True).agg(
        loans=("loan_id", "size"),
        principal_egp=("principal_egp", "sum"),
        mature=("mature", "sum"),
        defaults=("dpd30_default", "sum"),
        unpaid_at_extract=("unpaid_at_extract", "sum"),
    )
    result["immature"] = result.loans - result.mature
    n = result.mature.where(result.mature.gt(0))
    rate = result.defaults / n
    result["dpd30_pct"] = 100 * rate
    result["unpaid_at_extract_pct"] = 100 * result.unpaid_at_extract / result.loans

    # Wilson 95% interval for each mature default rate.
    z = 1.959963984540054
    denominator = 1 + z**2 / n
    center = (rate + z**2 / (2 * n)) / denominator
    margin = z * ((rate * (1 - rate) / n + z**2 / (4 * n**2)) ** .5) / denominator
    result["ci95_low_pct"] = 100 * (center - margin)
    result["ci95_high_pct"] = 100 * (center + margin)
    return result


monthly = credit_summary(q3, "month")
display(monthly.round(2))
print(f"Overall mature DPD30: {q3.dpd30_default.sum()}/{q3.mature.sum()} "
      f"({q3.dpd30_default.sum() / q3.mature.sum():.2%})")

# Same day-of-month range observable for September.
common_day = q3.loc[q3.mature, "disbursed_date"].max().day
matched = credit_summary(q3.loc[q3.disbursed_date.dt.day.le(common_day)], "month")
rate_columns = ["mature", "defaults", "dpd30_pct", "ci95_low_pct", "ci95_high_pct"]
print(f"Comparable disbursal windows: days 1–{common_day} of each month")
display(matched[rate_columns].round(2))

,loans,principal_egp,mature,defaults,unpaid_at_extract,immature,dpd30_pct,unpaid_at_extract_pct,ci95_low_pct,ci95_high_pct
month,,,,,,,,,,
2026-07,260,341500,260,17,17,0,6.54,6.54,4.12,10.22
2026-08,427,548000,427,25,25,0,5.85,5.85,4.00,8.50
2026-09,623,826000,353,34,73,270,9.63,11.72,6.97,13.16


Overall mature DPD30: 76/1040 (7.31%)
Comparable disbursal windows: days 1–22 of each month


,mature,defaults,dpd30_pct,ci95_low_pct,ci95_high_pct
month,,,,,
2026-07,139,8,5.76,2.94,10.95
2026-08,288,16,5.56,3.45,8.83
2026-09,353,34,9.63,6.97,13.16


## Risk by channel and booked loan size

Rates by channel and principal are descriptive. Denominators and uncertainty intervals help spot small cells. Differences can reflect borrower or underwriting mix. For unrecovered principal, apply observed repayments to principal first and floor the result at zero. This is an extract-date proxy because payments are recorded cumulatively without a principal/fee allocation or recovery ledger.

In [3]:
mature = q3.loc[q3.mature].copy()
print("Mature default rates by disbursal month and acquisition channel")
display(credit_summary(mature, ["month", "channel"])[rate_columns].round(2))
print("Mature default rates by booked principal, pooled Q3")
display(credit_summary(mature, "principal_egp")[rate_columns].round(2))
print("Mature default rates by disbursal month and booked principal")
display(credit_summary(mature, ["month", "principal_egp"])[rate_columns].round(2))

mature["unrecovered_principal_proxy_egp"] = (
    mature.principal_egp.sub(mature.amount_repaid_egp).clip(lower=0)
    .where(mature.dpd30_default, 0)
)
loss = mature.groupby("month").agg(
    mature_principal_egp=("principal_egp", "sum"),
    unrecovered_principal_proxy_egp=("unrecovered_principal_proxy_egp", "sum"),
)
loss["unrecovered_share_pct"] = 100 * loss.unrecovered_principal_proxy_egp / loss.mature_principal_egp
display(loss.round(2))

Mature default rates by disbursal month and acquisition channel


mature  defaults  dpd30_pct  ci95_low_pct  ci95_high_pct
month   channel                                                               
2026-07 Direct            20         2      10.00          2.79          30.10
        Google Ads        81         2       2.47          0.68           8.56
        Meta              52         5       9.62          4.18          20.61
        Organic           17         2      11.76          3.29          34.34
        Referral          43         2       4.65          1.28          15.46
        TikTok            16         3      18.75          6.59          43.01
        Unattributed      31         1       3.23          0.57          16.19
2026-08 Direct            19         2      10.53          2.94          31.39
        Google Ads       142         5       3.52          1.51           7.98
        Meta             107        16      14.95          9.42          22.92
        Organic           24         0       0.00          0.00          13.80
        Referral          67         0       0.00          0.00           5.42
        TikTok            21         0       0.00         -0.00          15.46
        Unattributed      47         2       4.26          1.17          14.25
2026-09 Direct            12         0       0.00          0.00          24.25
        Google Ads        90         6       6.67          3.09          13.79
        Meta             111        20      18.02         11.98          26.20
        Organic           15         1       6.67          1.19          29.82
        Referral          64         0       0.00          0.00           5.66
        TikTok            34         6      17.65          8.35          33.51
        Unattributed      27         1       3.70          0.66          18.28

Mature default rates by booked principal, pooled Q3


,mature,defaults,dpd30_pct,ci95_low_pct,ci95_high_pct
principal_egp,,,,,
500,279,15,5.38,3.28,8.68
1000,307,23,7.49,5.04,10.99
1500,216,24,11.11,7.58,16.00
2000,144,10,6.94,3.82,12.31
3000,94,4,4.26,1.67,10.44


Mature default rates by disbursal month and booked principal


mature  defaults  dpd30_pct  ci95_low_pct  \
month   principal_egp                                              
2026-07 500                63         0       0.00          0.00   
        1000               81         5       6.17          2.67   
        1500               56         5       8.93          3.87   
        2000               35         4      11.43          4.54   
        3000               25         3      12.00          4.17   
2026-08 500               116         4       3.45          1.35   
        1000              132         7       5.30          2.59   
        1500               78        10      12.82          7.12   
        2000               62         4       6.45          2.54   
        3000               39         0       0.00         -0.00   
2026-09 500               100        11      11.00          6.25   
        1000               94        11      11.70          6.66   
        1500               82         9      10.98          5.88   
        2000               47         2       4.26          1.17   
        3000               30         1       3.33          0.59   

                       ci95_high_pct  
month   principal_egp                 
2026-07 500                     5.75  
        1000                   13.65  
        1500                   19.26  
        2000                   25.95  
        3000                   29.96  
2026-08 500                     8.53  
        1000                   10.54  
        1500                   22.02  
        2000                   15.45  
        3000                    8.97  
2026-09 500                    18.63  
        1000                   19.75  
        1500                   19.56  
        2000                   14.25  
        3000                   16.67

,mature_principal_egp,unrecovered_principal_proxy_egp,unrecovered_share_pct
month,,,
2026-07,341500,26826.96,7.86
2026-08,548000,29472.36,5.38
2026-09,451000,33532.08,7.44


## Scope of the proposed limit cut

The extract provides booked principal and leaves assigned limits, requested amounts, approval decisions and underwriting scores unobserved. To illustrate scale, apply a single cap equal to 70% of the maximum observed booked principal. Hold loan counts fixed. This measures a mechanical exposure change under the illustrative cap. The impact of cutting every individual limit on approvals, take-up and defaults remains unknown.

In [4]:
cap = .70 * q3.principal_egp.max()
capped_principal = q3.principal_egp.clip(upper=cap)
cap_scope = pd.Series({
    "illustrative_cap_egp": cap,
    "historical_bookings_above_cap": q3.principal_egp.gt(cap).sum(),
    "bookings_above_cap_pct": 100 * q3.principal_egp.gt(cap).mean(),
    "historical_principal_egp": q3.principal_egp.sum(),
    "principal_under_illustrative_cap_egp": capped_principal.sum(),
    "mechanical_principal_reduction_egp": q3.principal_egp.sum() - capped_principal.sum(),
}, name="value")
display(cap_scope.to_frame().round(2))

,value
illustrative_cap_egp,2100.0
historical_bookings_above_cap,131.0
bookings_above_cap_pct,10.0
historical_principal_egp,1715500.0
principal_under_illustrative_cap_egp,1597600.0
mechanical_principal_reduction_egp,117900.0


## Recommendation for Finance

**Keep existing maximums for now and test a targeted policy.** Mature DPD30 was **6.54% in July, 5.85% in August and 9.63% in September**. September is higher among the loans we can observe, and **270 of 623** September loans are still immature. Restricting all three cohorts to days 1–22 gives **5.76%**, **5.56%**, and **9.63%**. The size of the July-to-September increase depends on which disbursals are compared.

Ask Finance for the source of the approximately 12% figure. The extract gives **11.72%** unpaid at extract across all September originations, and **9.63%** DPD30 among mature September loans. Refresh September once the full cohort matures. Review borrower and approval mix using application, score and limit data before selecting a segment.

Pilot a lower limit or staged increases in a documented higher-risk segment with a comparable control group. Track mature DPD30, unrecovered principal per EGP disbursed, take-up, approvals and fee revenue, with a full follow-up window. Use the results to decide whether to expand the limit policy.